In [ ]:
!pip install -q huggingface_hub datasets torch tqdm

In [ ]:
import os
import sys
import torch
import json
from torch.utils.data import DataLoader
from huggingface_hub import HfApi, hf_hub_download, notebook_login

# Add src to path if not already there
if 'src' not in sys.path:
    sys.path.append(os.path.abspath('src'))

from data import get_jefferson_text, preprocess, build_vocab, TrigramDataset
from models import CountTrigramModel, NeuralTrigramModel
from train import train_neural_model
from decode import generate_greedy, generate_top_k, generate_nucleus, generate_beam_search
from eval import calculate_perplexity_count, calculate_perplexity_neural

## 1. Dataset Preparation

In [ ]:
print("Loading Thomas Jefferson speeches...")
text = get_jefferson_text()
tokens = preprocess(text)

# Split into train and test
split_idx = int(len(tokens) * 0.9)
train_tokens = tokens[:split_idx]
test_tokens = tokens[split_idx:]

vocab, word2idx, idx2word = build_vocab(train_tokens)
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
print(f"Train tokens: {len(train_tokens)}")
print(f"Test tokens: {len(test_tokens)}")

## 2. Model Development

### 2.1 Count-based Trigram Model

In [ ]:
train_idx = [word2idx[w] for w in train_tokens]
test_idx = [word2idx[w] for w in test_tokens if w in word2idx]

count_model = CountTrigramModel(vocab_size=vocab_size, add_k=0.1)
count_model.train(train_idx)

ppl_count = calculate_perplexity_count(count_model, test_idx)
print(f"Count Model Test Perplexity: {ppl_count:.4f}")

### 2.2 Neural Trigram Model

In [ ]:
train_dataset = TrigramDataset(train_tokens, word2idx)
val_split = int(len(train_dataset) * 0.9)
train_ds, val_ds = torch.utils.data.random_split(train_dataset, [val_split, len(train_dataset) - val_split])
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
nn_model = NeuralTrigramModel(vocab_size=vocab_size, embed_size=64, hidden_size=128)
nn_model = train_neural_model(nn_model, train_loader, val_loader, epochs=50, patience=5, lr=1e-3, device=device)

test_dataset = TrigramDataset([w for w in test_tokens if w in word2idx], word2idx)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)
ppl_nn = calculate_perplexity_neural(nn_model, test_loader, device=device)
print(f"Neural Model Test Perplexity: {ppl_nn:.4f}")

## 3. Upload to Hugging Face

We will upload the `neural_model.pth` and the vocabulary (`vocab.json`).

In [ ]:
# Save locally first
torch.save(nn_model.state_dict(), 'nn_model.pth')
with open('vocab.json', 'w') as f:
    json.dump({'word2idx': word2idx, 'idx2word': {str(k): v for k, v in idx2word.items()}}, f)

print("Model and vocab saved locally.")

In [ ]:
repo_id = "bdanko/n-gram-modeling"
api = HfApi()

try:
    api.upload_file(
        path_or_fileobj="nn_model.pth",
        path_in_repo="nn_model.pth",
        repo_id=repo_id
    )
    api.upload_file(
        path_or_fileobj="vocab.json",
        path_in_repo="vocab.json",
        repo_id=repo_id
    )
    print(f"Successfully uploaded to {repo_id}")
except Exception as e:
    print(f"Upload failed: {e}")

## 4. Evaluation Using the Uploaded Model

In [ ]:
print("Downloading model from Hugging Face...")
hf_model_path = hf_hub_download(repo_id=repo_id, filename="nn_model.pth")
hf_vocab_path = hf_hub_download(repo_id=repo_id, filename="vocab.json")

with open(hf_vocab_path, 'r') as f:
    hf_vocab_data = json.load(f)
    hf_word2idx = hf_vocab_data['word2idx']
    hf_idx2word = {int(k): v for k, v in hf_vocab_data['idx2word'].items()}

# Initialize new model instance and load weights
loaded_nn_model = NeuralTrigramModel(vocab_size=len(hf_word2idx), embed_size=64, hidden_size=128)
loaded_nn_model.load_state_dict(torch.load(hf_model_path, weights_only=True))
loaded_nn_model.eval()

print("Model loaded successfully from HF.")

In [ ]:
w1_word, w2_word = "fellow", "citizens"
w1, w2 = hf_word2idx[w1_word], hf_word2idx[w2_word]

print(f"Seed: '{w1_word} {w2_word}'")

print("\nGreedy")
print(generate_greedy(loaded_nn_model, w1, w2, hf_word2idx, hf_idx2word, num_words=80))

print("\nBeam Search (width=3)")
print(generate_beam_search(loaded_nn_model, w1, w2, hf_word2idx, hf_idx2word, num_words=80, beam_width=3))

print("\nTop-K Sampling (K=5)")
print(generate_top_k(loaded_nn_model, w1, w2, hf_word2idx, hf_idx2word, num_words=80, k=5))

print("\nNucleus Sampling (p=0.9)")
print(generate_nucleus(loaded_nn_model, w1, w2, hf_word2idx, hf_idx2word, num_words=80, p=0.9))